In [3]:
import os
path_to_main_experiment_folder =  "/home/pablo.canosa/Datos/pruebas ctcomp/pruebas a80 rios"
folders = os.listdir(path_to_main_experiment_folder)
# sort folders by name
folders.sort()
print(folders)

['Mask2Former_rios_rgb_FAPN_1', 'Mask2Former_rios_rgb_FAPN_2', 'Mask2Former_rios_rgb_FAPN_3', 'Mask2Former_rios_rgb_FAPN_4', 'Mask2Former_rios_rgb_FAPN_5', 'Mask2Former_rios_rgb_Standard_2_2', 'Mask2Former_rios_rgb_Standard_2_3', 'Mask2Former_rios_rgb_Standard_2_4', 'Mask2Former_rios_rgb_Standard_2_5', 'Mask2Former_rios_rgb_standard_2_1', 'Mask2former_FPN_RGB_1', 'Mask2former_FPN_RGB_1.log', 'Mask2former_FPN_RGB_2', 'Mask2former_FPN_RGB_3', 'Mask2former_FPN_RGB_4', 'Mask2former_FPN_RGB_5']


In [1]:
# Example of the last line of test in the JSON file
# {"iteration": 3100, "sem_seg/ACC-Asphalt": 96.35847410166578, "sem_seg/ACC-Bare soil": 92.31057306755174, 
#  "sem_seg/ACC-Concrete": 77.4352285527954, "sem_seg/ACC-Eucalyptus": 96.28301180673355, 
#  "sem_seg/ACC-Meadows": 93.54667494011116, "sem_seg/ACC-Native trees": 95.39228937828993, 
#  "sem_seg/ACC-Pines": 88.90903744363851, "sem_seg/ACC-Rock": 79.42387311213481, 
#  "sem_seg/ACC-Tiles": 99.24320605435156, "sem_seg/ACC-Water": 87.21770881618646, 
#  "sem_seg/ACC-unlabeled": NaN, "sem_seg/BoundaryIoU-Asphalt": 3.907173045820504, 
#  "sem_seg/BoundaryIoU-Bare soil": 5.231094844308878, "sem_seg/BoundaryIoU-Concrete": 20.923323219449376, 
#  "sem_seg/BoundaryIoU-Eucalyptus": 59.10558158595448, "sem_seg/BoundaryIoU-Meadows": 36.83782604854562, 
#  "sem_seg/BoundaryIoU-Native trees": 13.552497669622232, "sem_seg/BoundaryIoU-Pines": 1.529448628492772, 
#  "sem_seg/BoundaryIoU-Rock": 4.480138303435533, "sem_seg/BoundaryIoU-Tiles": 0.6596654727544597, 
#  "sem_seg/BoundaryIoU-Water": 0.7163346552408261, "sem_seg/BoundaryIoU-unlabeled": 69.0929941534061, 
#  "sem_seg/IoU-Asphalt": 90.60404739307474, "sem_seg/IoU-Bare soil": 84.07540027301111, 
#  "sem_seg/IoU-Concrete": 70.96633975386455, "sem_seg/IoU-Eucalyptus": 89.38051683150977, 
#  "sem_seg/IoU-Meadows": 90.44299247206455, "sem_seg/IoU-Native trees": 88.83477738564032, 
#  "sem_seg/IoU-Pines": 87.13507593878786, "sem_seg/IoU-Rock": 75.27162199502625, 
#  "sem_seg/IoU-Tiles": 77.57461683248185, "sem_seg/IoU-Water": 76.8358384318895, 
#  "sem_seg/IoU-unlabeled": NaN, "sem_seg/fwIoU": 88.87666137363179, 
#  "sem_seg/mACC": 90.6120077273459, "sem_seg/mIoU": 83.11212273073505, 
#  "sem_seg/min(IoU, B-Iou)-Asphalt": 3.907173045820504, "sem_seg/min(IoU, B-Iou)-Bare soil": 5.231094844308878, 
#  "sem_seg/min(IoU, B-Iou)-Concrete": 20.923323219449376, "sem_seg/min(IoU, B-Iou)-Eucalyptus": 59.10558158595448, 
#  "sem_seg/min(IoU, B-Iou)-Meadows": 36.83782604854562, "sem_seg/min(IoU, B-Iou)-Native trees": 13.552497669622232, 
#  "sem_seg/min(IoU, B-Iou)-Pines": 1.529448628492772, "sem_seg/min(IoU, B-Iou)-Rock": 4.480138303435533, 
#  "sem_seg/min(IoU, B-Iou)-Tiles": 0.6596654727544597, "sem_seg/min(IoU, B-Iou)-Water": 0.7163346552408261, 
#  "sem_seg/min(IoU, B-Iou)-unlabeled": NaN, "sem_seg/pACC": 94.08328596871515}

# steps to read JSON
#1. Open the file and read it from the ending
#2. Gather the data from the last line that contains "sem_seg/ACC-XXXX" and "sem_seg/IoU-XXXX"
#3. Store the data in a dictionary as {class_name: (accuracy, iou)}

import json
import os
import math

def sanitize_class_metrics_from_json(path_to_metrics_json):
    """
    Reads a Detectron2 metrics.json file, finds the last evaluation entry,
    and returns a dictionary of {class_name: (accuracy, iou, boundary_iou, min_iou)}.
    """
    if not os.path.exists(path_to_metrics_json):
        print(f"Error: File not found at {path_to_metrics_json}")
        return {}

    last_valid_metrics = None

    with open(path_to_metrics_json, 'r') as f:
        # Read all lines
        lines = f.readlines()
        
        # 1. Iterate backwards to find the last line containing class-wise metrics
        for line in reversed(lines):
            line = line.strip()
            if not line:
                continue
            
            try:
                data = json.loads(line)
                
                # Check if this specific line contains the semantic segmentation class breakdown
                if any(k.startswith("sem_seg/ACC-") for k in data.keys()):
                    last_valid_metrics = data
                    break
            except json.JSONDecodeError:
                continue
    
    if last_valid_metrics is None:
        print("No valid semantic segmentation class metrics found in the file.")
        return {}

    results = {}

    # 2. & 3. Gather data and store in dictionary
    for key, value in last_valid_metrics.items():
        if key.startswith("sem_seg/ACC-"):
            # Extract class name (remove prefix)
            class_name = key.replace("sem_seg/ACC-", "")
            
            # 1. Get Accuracy
            acc = value
            
            # 2. Get IoU
            iou_key = f"sem_seg/IoU-{class_name}"
            iou = last_valid_metrics.get(iou_key, None)

            # 3. Get Boundary IoU
            b_iou_key = f"sem_seg/BoundaryIoU-{class_name}"
            b_iou = last_valid_metrics.get(b_iou_key, None)

            # 4. Get Min(IoU, B-IoU)
            # Note: based on your json snippet, the key usually has 'B-Iou' (lowercase u)
            min_iou_key = f"sem_seg/min(IoU, B-Iou)-{class_name}"
            min_iou = last_valid_metrics.get(min_iou_key, None)
            
            # Store in the dictionary as a 4-tuple
            results[class_name] = (acc, iou, b_iou, min_iou)

    return results

def compute_average_metrics(list_of_paths):
    metrics = []
    for metrics_paths in list_of_paths:
        path_to_metrics_json = os.path.join(metrics_paths, "metrics.json")
        class_metrics = sanitize_class_metrics_from_json(path_to_metrics_json)
        if class_metrics: # Only append if valid metrics were found
            metrics.append(class_metrics)
    
    if not metrics:
        return {}

    # Compute average metrics per class
    average_metrics = {}
    
    # Use keys from the first valid metric dict found
    all_classes = metrics[0].keys()

    for class_name in all_classes:
        total_acc = 0
        total_iou = 0
        total_b_iou = 0
        total_min_iou = 0
        
        # Counters for valid (non-None) values
        count_acc = 0
        count_iou = 0
        count_b_iou = 0
        count_min_iou = 0

        for metric in metrics:
            if class_name in metric:
                acc, iou, b_iou, min_iou = metric[class_name]
                
                # Helper to accumulate safely
                if acc is not None and not math.isnan(acc):
                    total_acc += acc
                    count_acc += 1
                if iou is not None and not math.isnan(iou):
                    total_iou += iou
                    count_iou += 1
                if b_iou is not None and not math.isnan(b_iou):
                    total_b_iou += b_iou
                    count_b_iou += 1
                if min_iou is not None and not math.isnan(min_iou):
                    total_min_iou += min_iou
                    count_min_iou += 1

        # Calculate averages, default to None (or NaN) if no valid data found
        avg_acc = total_acc / count_acc if count_acc > 0 else float('nan')
        avg_iou = total_iou / count_iou if count_iou > 0 else float('nan')
        avg_b_iou = total_b_iou / count_b_iou if count_b_iou > 0 else float('nan')
        avg_min_iou = total_min_iou / count_min_iou if count_min_iou > 0 else float('nan')

        average_metrics[class_name] = (avg_acc, avg_iou, avg_b_iou, avg_min_iou)

    return average_metrics

def tabulate_metrics_to_markdown(metrics):
    """
    Converts a dictionary of {class_name: (accuracy, iou, b_iou, min_iou)} into a Markdown table string.
    """
    if not metrics:
        return "No metrics available to tabulate."

    # Define the header with 4 metric columns
    table = "| Class Name | Accuracy (%) | IoU (%) | Boundary IoU (%) | Min IoU (%) |\n"
    table += "| :--- | :---: | :---: | :---: | :---: |\n"

    # Helper function to format numbers
    def format_val(val):
        try:
            if val is None or (isinstance(val, float) and math.isnan(val)):
                return "NaN"
            return f"{val:.2f}"
        except (ValueError, TypeError):
            return str(val)

    # Sort by class name for a clean table
    for class_name in sorted(metrics.keys()):
        # Unpack the 4 values
        values = metrics[class_name]
        
        # Handle cases where dictionary might still have old 2-tuple format (defensive coding)
        if len(values) == 4:
            acc, iou, b_iou, min_iou = values
        else:
            acc, iou = values
            b_iou, min_iou = None, None

        acc_str = format_val(acc)
        iou_str = format_val(iou)
        b_iou_str = format_val(b_iou)
        min_iou_str = format_val(min_iou)
        
        table += f"| {class_name} | {acc_str} | {iou_str} | {b_iou_str} | {min_iou_str} |\n"

    return table

In [8]:
import os
path_to_main_experiment_folder =  "/home/pablo.canosa/Datos/pruebas ctcomp/pruebas a80 rios"
folders = os.listdir(path_to_main_experiment_folder)
# sort folders by name
folders.sort()
print(folders)

path_to_test_json = "/home/pablo.canosa/Datos/pruebas ctcomp/pruebas a80 rios/Mask2Former_rios_rgb_FAPN_1/metrics.json"
metrics = sanitize_class_metrics_from_json(path_to_test_json)
print(metrics)

experiment_path = "/home/pablo.canosa/Datos/pruebas ctcomp/pruebas a80 rios"
list_of_experiments = [os.path.join(experiment_path, folder) for folder in folders]
fapn_list = [path for path in list_of_experiments if "FAPN" in path]
Standard_list = [path for path in list_of_experiments if "Standard" in path]
FPN_list = [path for path in list_of_experiments if "FPN" in path]
print("FAPN experiments:", fapn_list)
print("Standard experiments:", Standard_list)
print("FPN experiments:", FPN_list)
average_fapn_metrics = compute_average_metrics(fapn_list)
average_standard_metrics = compute_average_metrics(Standard_list)
average_fpn_metrics = compute_average_metrics(FPN_list)

['Mask2Former_rios_rgb_FAPN_1', 'Mask2Former_rios_rgb_FAPN_2', 'Mask2Former_rios_rgb_FAPN_3', 'Mask2Former_rios_rgb_FAPN_4', 'Mask2Former_rios_rgb_FAPN_5', 'Mask2Former_rios_rgb_Standard_2_1', 'Mask2Former_rios_rgb_Standard_2_2', 'Mask2Former_rios_rgb_Standard_2_3', 'Mask2Former_rios_rgb_Standard_2_4', 'Mask2Former_rios_rgb_Standard_2_5', 'Mask2former_FPN_RGB_1', 'Mask2former_FPN_RGB_2', 'Mask2former_FPN_RGB_3', 'Mask2former_FPN_RGB_4', 'Mask2former_FPN_RGB_5']
{'Asphalt': (96.35847410166578, 90.60404739307474, 3.907173045820504, 3.907173045820504), 'Bare soil': (92.31057306755174, 84.07540027301111, 5.231094844308878, 5.231094844308878), 'Concrete': (77.4352285527954, 70.96633975386455, 20.923323219449376, 20.923323219449376), 'Eucalyptus': (96.28301180673355, 89.38051683150977, 59.10558158595448, 59.10558158595448), 'Meadows': (93.54667494011116, 90.44299247206455, 36.83782604854562, 36.83782604854562), 'Native trees': (95.39228937828993, 88.83477738564032, 13.552497669622232, 13.552

In [10]:
md_table_fapn = tabulate_metrics_to_markdown(average_fapn_metrics)
print("FAPN Average Metrics for rios dataset:\n", md_table_fapn)
md_table_standard = tabulate_metrics_to_markdown(average_standard_metrics)
print("Standard Average Metrics for rios dataset:\n", md_table_standard)
md_table_fpn = tabulate_metrics_to_markdown(average_fpn_metrics)
print("FPN Average Metrics for rios dataset:\n", md_table_fpn)

FAPN Average Metrics for rios dataset:
 | Class Name | Accuracy (%) | IoU (%) | Boundary IoU (%) | Min IoU (%) |
| :--- | :---: | :---: | :---: | :---: |
| Asphalt | 97.78 | 87.27 | 3.95 | 3.95 |
| Bare soil | 92.35 | 84.17 | 5.82 | 5.82 |
| Concrete | 73.82 | 68.59 | 20.89 | 20.89 |
| Eucalyptus | 95.84 | 88.37 | 58.10 | 58.10 |
| Meadows | 93.15 | 90.06 | 33.42 | 33.42 |
| Native trees | 95.01 | 87.82 | 12.93 | 12.93 |
| Pines | 87.00 | 80.35 | 1.08 | 1.08 |
| Rock | 84.85 | 79.54 | 4.53 | 4.53 |
| Tiles | 99.69 | 92.13 | 0.79 | 0.79 |
| Water | 85.42 | 79.05 | 0.77 | 0.77 |
| unlabeled | NaN | NaN | 69.24 | NaN |

Standard Average Metrics for rios dataset:
 | Class Name | Accuracy (%) | IoU (%) | Boundary IoU (%) | Min IoU (%) |
| :--- | :---: | :---: | :---: | :---: |
| Asphalt | 97.45 | 86.08 | 4.01 | 4.01 |
| Bare soil | 92.57 | 84.10 | 5.91 | 5.91 |
| Concrete | 43.22 | 41.12 | 20.19 | 20.19 |
| Eucalyptus | 95.26 | 89.25 | 58.97 | 58.97 |
| Meadows | 94.18 | 90.79 | 36.55 | 36.

In [11]:
path_to_lados_exp = "/home/pablo.canosa/Datos/pruebas ctcomp/pruebas_LADOS/outputs_LADOS"
folders = os.listdir(path_to_lados_exp)
list_of_experiments = [os.path.join(path_to_lados_exp, folder) for folder in folders]
print(list_of_experiments)
fapn_list = [path for path in list_of_experiments if "fapn" in path]
Standard_list = [path for path in list_of_experiments if "std" in path]
FPN_list = [path for path in list_of_experiments if "FPN" in path]
print("FAPN experiments:", fapn_list)
print("Standard experiments:", Standard_list)
print("FPN experiments:", FPN_list)
average_fapn_metrics = compute_average_metrics(fapn_list)
average_standard_metrics = compute_average_metrics(Standard_list)
average_fpn_metrics = compute_average_metrics(FPN_list)

['/home/pablo.canosa/Datos/pruebas ctcomp/pruebas_LADOS/outputs_LADOS/lados_std_4', '/home/pablo.canosa/Datos/pruebas ctcomp/pruebas_LADOS/outputs_LADOS/lados_FPN_3', '/home/pablo.canosa/Datos/pruebas ctcomp/pruebas_LADOS/outputs_LADOS/lados_std_5', '/home/pablo.canosa/Datos/pruebas ctcomp/pruebas_LADOS/outputs_LADOS/lados_fapn_5', '/home/pablo.canosa/Datos/pruebas ctcomp/pruebas_LADOS/outputs_LADOS/lados_fapn_1', '/home/pablo.canosa/Datos/pruebas ctcomp/pruebas_LADOS/outputs_LADOS/lados_fapn_4', '/home/pablo.canosa/Datos/pruebas ctcomp/pruebas_LADOS/outputs_LADOS/lados_FPN_5', '/home/pablo.canosa/Datos/pruebas ctcomp/pruebas_LADOS/outputs_LADOS/lados_fapn_3', '/home/pablo.canosa/Datos/pruebas ctcomp/pruebas_LADOS/outputs_LADOS/lados_std_2', '/home/pablo.canosa/Datos/pruebas ctcomp/pruebas_LADOS/outputs_LADOS/lados_FPN_2', '/home/pablo.canosa/Datos/pruebas ctcomp/pruebas_LADOS/outputs_LADOS/lados_std_1', '/home/pablo.canosa/Datos/pruebas ctcomp/pruebas_LADOS/outputs_LADOS/lados_fapn_2'

In [12]:
md_table_standard = tabulate_metrics_to_markdown(average_standard_metrics)
print("Standard Average Metrics for LADOS dataset:\n", md_table_standard)
md_table_fapn = tabulate_metrics_to_markdown(average_fapn_metrics)
print("FAPN Average Metrics for LADOS dataset:\n", md_table_fapn)
md_table_fpn = tabulate_metrics_to_markdown(average_fpn_metrics)
print("FPN Average Metrics for LADOS dataset:\n", md_table_fpn)

Standard Average Metrics for LADOS dataset:
 | Class Name | Accuracy (%) | IoU (%) | Boundary IoU (%) | Min IoU (%) |
| :--- | :---: | :---: | :---: | :---: |
| Background | NaN | NaN | 86.70 | NaN |
| Emulsion | 94.19 | 87.81 | 32.58 | 32.58 |
| Oil | 95.42 | 89.24 | 38.85 | 38.85 |
| Oil-platform | 77.12 | 71.49 | 5.00 | 5.00 |
| Sheen | 86.05 | 80.02 | 21.37 | 21.37 |
| Ship | 83.21 | 78.69 | 1.84 | 1.84 |

FAPN Average Metrics for LADOS dataset:
 | Class Name | Accuracy (%) | IoU (%) | Boundary IoU (%) | Min IoU (%) |
| :--- | :---: | :---: | :---: | :---: |
| Background | NaN | NaN | 88.03 | NaN |
| Emulsion | 94.79 | 87.84 | 38.76 | 38.76 |
| Oil | 95.21 | 89.49 | 42.80 | 42.80 |
| Oil-platform | 81.41 | 80.61 | 5.05 | 5.05 |
| Sheen | 85.49 | 79.40 | 20.15 | 20.15 |
| Ship | 84.07 | 79.50 | 1.93 | 1.93 |

FPN Average Metrics for LADOS dataset:
 | Class Name | Accuracy (%) | IoU (%) | Boundary IoU (%) | Min IoU (%) |
| :--- | :---: | :---: | :---: | :---: |
| Background | NaN | N

# Five Billion Pixels

In [13]:
path_to_lados_exp = "/home/pablo.canosa/Datos/pruebas ctcomp/output_gaofen_rgb"
folders = os.listdir(path_to_lados_exp)
list_of_experiments = [os.path.join(path_to_lados_exp, folder) for folder in folders]
print(list_of_experiments)
fapn_list = [path for path in list_of_experiments if "FAPN" in path]
Standard_list = [path for path in list_of_experiments if "MSD" in path]
FPN_list = [path for path in list_of_experiments if "FPN" in path]
print("FAPN experiments:", fapn_list)
print("Standard experiments:", Standard_list)
print("FPN experiments:", FPN_list)
average_fapn_metrics = compute_average_metrics(fapn_list)
average_standard_metrics = compute_average_metrics(Standard_list)
average_fpn_metrics = compute_average_metrics(FPN_list)


['/home/pablo.canosa/Datos/pruebas ctcomp/output_gaofen_rgb/Mask2Former_MSD_2_2', '/home/pablo.canosa/Datos/pruebas ctcomp/output_gaofen_rgb/Mask2Former_FPN_2', '/home/pablo.canosa/Datos/pruebas ctcomp/output_gaofen_rgb/Mask2Former_MSD_2_5', '/home/pablo.canosa/Datos/pruebas ctcomp/output_gaofen_rgb/Mask2Former_FAPN_1', '/home/pablo.canosa/Datos/pruebas ctcomp/output_gaofen_rgb/Mask2Former_FAPN_3', '/home/pablo.canosa/Datos/pruebas ctcomp/output_gaofen_rgb/Mask2Former_FAPN_4', '/home/pablo.canosa/Datos/pruebas ctcomp/output_gaofen_rgb/Mask2Former_MSD_2_4', '/home/pablo.canosa/Datos/pruebas ctcomp/output_gaofen_rgb/Mask2Former_FAPN_5', '/home/pablo.canosa/Datos/pruebas ctcomp/output_gaofen_rgb/Mask2Former_MSD_2_1', '/home/pablo.canosa/Datos/pruebas ctcomp/output_gaofen_rgb/Mask2Former_MSD_2_3', '/home/pablo.canosa/Datos/pruebas ctcomp/output_gaofen_rgb/Mask2Former_FAPN_2', '/home/pablo.canosa/Datos/pruebas ctcomp/output_gaofen_rgb/Mask2Former_FPN_5', '/home/pablo.canosa/Datos/pruebas ct

In [14]:
md_table_standard = tabulate_metrics_to_markdown(average_standard_metrics)
print("Standard Average Metrics for FBP dataset:\n", md_table_standard)
md_table_fapn = tabulate_metrics_to_markdown(average_fapn_metrics)
print("FAPN Average Metrics for FBP dataset:\n", md_table_fapn)
md_table_fpn = tabulate_metrics_to_markdown(average_fpn_metrics)
print("FPN Average Metrics for FBP dataset:\n", md_table_fpn)

Standard Average Metrics for FBP dataset:
 | Class Name | Accuracy (%) | IoU (%) | Boundary IoU (%) | Min IoU (%) |
| :--- | :---: | :---: | :---: | :---: |
| airport | 67.85 | 57.20 | 0.00 | 0.00 |
| arbor forest | 93.75 | 91.07 | 22.71 | 22.71 |
| artificial meadow | 59.50 | 45.09 | 0.00 | 0.00 |
| bareland | 86.53 | 78.04 | 0.00 | 0.00 |
| dry cropland | 79.59 | 72.27 | 10.28 | 10.28 |
| fish pond | 86.68 | 71.39 | 0.00 | 0.00 |
| garden land | 53.53 | 41.71 | 2.74 | 2.74 |
| industrial area | 80.27 | 71.48 | 9.78 | 9.78 |
| irrigated field | 96.41 | 90.72 | 36.25 | 36.25 |
| lake | 91.58 | 74.78 | 0.00 | 0.00 |
| natural meadow | 90.69 | 83.35 | 24.08 | 24.08 |
| overpass | 68.91 | 61.41 | 0.00 | 0.00 |
| paddy field | 77.47 | 68.76 | 6.56 | 6.56 |
| park | 32.08 | 30.01 | 6.08 | 6.08 |
| pond | 41.61 | 32.62 | 0.00 | 0.00 |
| railway station | 59.81 | 38.49 | 0.00 | 0.00 |
| river | 69.41 | 61.63 | 0.00 | 0.00 |
| road | 83.14 | 68.30 | 0.00 | 0.00 |
| rural residential | 85.78 | 